# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nazama-tech/Flyrank-Ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import duckdb as duckbd
import pandas as pd

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

con = duckbd.connect()
con.execute(f"""CREATE SECRET(TYPE huggingface, TOKEN '{HF_TOKEN}')""")

print("Connected Successfully!")

Connected Successfully!


In [5]:
rel = "hf://datasets/FlyRank/internship-warehouse"
print("Warehouse path ready!")

Warehouse path ready!


In [6]:
con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [7]:
# descriptive statistics and percentiles.
sample = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    scroll_events
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE gsc_data_available IS TRUE
LIMIT 100000
""").df()

sample.describe()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
count,100000.000000,100000.000000,99999.000000,100000.0,100000.0
mean,17.000510,0.114610,28.890156,0.0,0.0
std,29.472032,0.503147,22.812135,0.0,0.0
min,1.000000,0.000000,0.000000,0.0,0.0
25%,3.000000,0.000000,9.298438,0.0,0.0
50%,8.000000,0.000000,22.000000,0.0,0.0
75%,20.000000,0.000000,43.750000,0.0,0.0
max,945.000000,33.000000,204.000000,0.0,0.0


Distribution observations: GSC impressions are strongly right-skewed, with a median of 8 impressions compared with a maximum of 945, indicating a long tail of high-volume rows. GSC clicks are highly sparse and zero-inflated: at least 75% of sampled rows have zero clicks. Average position has a wide spread, so CTR should not be compared across all positions without accounting for ranking position. GA4 sessions and scroll events were all zero in this sample, likely because the sample was filtered only for GSC availability; GA4 availability needs to be checked separately

In [8]:
# query for available Ga4
sample[[
    'ga4_sessions',
    'scroll_events'
]].describe()

,ga4_sessions,scroll_events
count,100000.0,100000.0
mean,0.0,0.0
std,0.0,0.0
min,0.0,0.0
25%,0.0,0.0
50%,0.0,0.0
75%,0.0,0.0
max,0.0,0.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [10]:
#

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.